# Tropycal quickstart — one Atlantic season

Fetch tropical-cyclone **best tracks** for the 2005 North Atlantic season from HURDAT2 through the `earthlens` Tropycal backend and plot them coloured by Saffir-Simpson category.

Tropycal is a `vector` backend: `download()` returns a pyramids `FeatureCollection` (a `geopandas.GeoDataFrame`). The default geometry is one `Point` per 6-hourly fix.

> **First-load cost.** The first query of a basin downloads and parses its whole best-track file (a few seconds for HURDAT, longer for IBTrACS). Needs `pip install earthlens[tropycal]`.

## Setup

Pull the imports together up front: `EarthLens` is the unified entry point, `matplotlib` renders the track scatter, and we prepare an output directory for the written best-track file.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from earthlens import EarthLens

OUT_DIR = Path('tropycal_output')
OUT_DIR.mkdir(exist_ok=True)

## Query

`variables` selects the **basin** (here `north_atlantic`). The whole season's storms are filtered to the date window and bbox at the fix level. The result is written to `tropycal_output/` and returned in memory.

### Build the request

Construct the `EarthLens` request on its own line so the source, basin, date window, and Gulf-of-Mexico bbox are easy to read and tweak.

In [ ]:
tracks = EarthLens(
    variables=['north_atlantic'],
    data_source='tropycal',
    start='2005-08-01',
    end='2005-09-15',
    aoi=[-98.0, 18.0, -80.0, 31.0],
    source='hurdat',
    path=str(OUT_DIR),
)

### Download the fixes

`download()` is kept on its own line so the live network call is a separate, re-runnable step. It is wrapped in a guard so the notebook stays safe under nbval-lax when run offline or without the `[tropycal]` extra.

In [ ]:
fixes = None
try:
    fixes = tracks.download(progress_bar=False)
    print(len(fixes), 'fixes')
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## Inspect the fixes

Peek at the first few rows — one per 6-hourly fix — to confirm the storm metadata, intensity (`vmax_kt`), category, and geometry came through, and check the CRS.

In [ ]:
if fixes is not None and len(fixes):
    cols = ['storm_id', 'name', 'time', 'vmax_kt', 'category', 'geometry']
    display(fixes[cols].head())
    print('CRS:', fixes.crs)

## Plot the track fixes coloured by category

Scatter every fix on a lon/lat axis, colouring each point by its Saffir-Simpson `category` so the storms' intensification across the Gulf stands out.

In [ ]:
if fixes is not None and len(fixes):
    fig, ax = plt.subplots(figsize=(8, 6))
    fixes.plot(
        ax=ax,
        column='category',
        cmap='YlOrRd',
        legend=True,
        markersize=25,
        edgecolor='k',
        linewidth=0.2,
    )
    ax.set_title('North Atlantic best-track fixes, Aug-Sep 2005 (Gulf of Mexico)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()